# Preprocessing — Member 3: Feature Selection
## Online Shopper Purchase Likelihood Prediction System
**IT3051 – Fundamentals of Data Mining | Mini Project 2026**

This notebook executes the **Feature Selection** stage of the preprocessing relay. It builds upon Member 1's engineered and log-transformed training handoff (`X_train_engineered.csv` and `y_train.csv`).

### Objectives & Scope:
1. **Bivariate & Multicollinearity Screening:** Screen high-correlation numeric pairs and screen categorical features using Chi-Square tests of independence (directly integrating Member 3 EDA findings).
2. **Cross-Validated Permutation Importance:** Quantify each feature's contribution to generalization performance (F1-score) on held-out validation folds using a balanced Random Forest estimator.
3. **Hypothesis-Driven Ablation Experiments:** Systematically test candidate drops using paired 5-fold Stratified Cross-Validation (testing `ExitRates`, collinear duplicates `ProductRelated`/`ProductRelated_Duration`, and negative-importance features).
4. **Sensitivity Analysis (Without `PageValues`):** Address Member 4's EDA finding regarding potential `PageValues` data leakage and timing availability by inspecting feature importance in early-session / cold-start conditions.
5. **Downstream Pipeline Handoff:** Output finalized lists of `SELECTED_NUMERIC_FEATURES` and `SELECTED_CATEGORICAL_FEATURES` for Member 2's encoding and scaling pipeline.

*Note: All feature evaluations, screenings, and importance calculations are performed strictly on the training set using Stratified 5-Fold Cross-Validation. The held-out test set remains completely untouched to prevent data leakage.*

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

from src.preprocessing.feature_selection import (
    categorical_chi2_screening,
    compare_feature_drop,
    correlation_pairs,
    cross_validated_permutation_importance,
    select_features_by_importance,
)

## 1. Load the Training Handoff

We load the deduplicated, split, feature-engineered, and log-transformed training set. We verify feature types and dimensions before performing selection.

In [2]:
X_train = pd.read_csv('../data/processed/X_train_engineered.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze('columns').astype(int)

categorical_columns = [
    'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType', 'Weekend', 'VisitorType_Weekend',
]
numeric_columns = [col for col in X_train.columns if col not in categorical_columns]
X_train_numeric = X_train[numeric_columns].copy()

print('Training rows:', len(X_train_numeric))
print('Numeric features evaluated:', len(numeric_columns))
print('Categorical features evaluated:', len(categorical_columns))
print(f'Target distribution (Revenue=1): {y_train.mean():.2%}')

Training rows: 9764
Numeric features evaluated: 13
Categorical features evaluated: 8
Target distribution (Revenue=1): 15.52%


## 2. Correlation Screening (Multicollinearity Analysis)

Pairwise Pearson correlation detects linear redundancy among numeric predictors. A threshold of $|r| \ge 0.80$ is used as an exploratory screening threshold on training data only.

> **Key Data Mining Principle:** Correlation only flags redundancy candidates; it does *not* automatically justify feature removal. Complex non-linear models (like Random Forests) can extract non-redundant split signals even from correlated features. Removal must be validated empirically.

In [3]:
correlated_pairs = correlation_pairs(X_train_numeric, threshold=0.80)
correlated_pairs

,feature_a,feature_b,correlation,absolute_correlation
0,ProductRelated,TotalPages,0.988947,0.988947
1,ProductRelated_Duration,TotalDuration,0.986111,0.986111
2,BounceRates,ExitRates,0.899991,0.899991
3,TotalPages,TotalDuration,0.853646,0.853646
4,ProductRelated,ProductRelated_Duration,0.852326,0.852326
5,ProductRelated_Duration,TotalPages,0.851255,0.851255
6,ProductRelated,TotalDuration,0.836330,0.836330
7,Informational,Informational_Duration,0.823713,0.823713


### Correlation Insights & Candidate Hypotheses:
1. **Near-Perfect Duplicate Pair:** `ProductRelated` and `TotalPages` ($r = 0.9889$) as well as `ProductRelated_Duration` and `TotalDuration` ($r = 0.9861$). Because `ProductRelated` constitutes over 90% of user browsing, engineered totals essentially replicate the raw columns. Keeping both introduces severe collinearity.
2. **Session Quality Pair:** `BounceRates` and `ExitRates` ($r = 0.8999$) are strongly correlated. EDA flagged this pair as candidate redundancy.
3. **Page Sub-types:** `Informational` and `Informational_Duration` ($r = 0.8237$) naturally correlate.

## 3. Categorical Feature Screening (Bridging EDA Chi-Square Tests)

In EDA (Member 3), Chi-Square tests of independence were conducted on the full dataset. Here, we formally integrate this statistical test into the preprocessing pipeline on the **training set** to evaluate which categorical features show statistically significant associations with `Revenue` ($p < 0.05$).

In [4]:
cat_screening_df = categorical_chi2_screening(X_train, y_train, categorical_columns=categorical_columns)
cat_screening_df

,feature,chi2,p_value,dof,significant
0,Month,299.467554,3.385508e-59,9,True
1,TrafficType,290.115311,5.170358e-51,18,True
2,VisitorType_Weekend,104.025547,7.483962e-21,5,True
3,VisitorType,89.475422,3.720994e-20,2,True
4,OperatingSystems,60.354881,1.282256e-10,7,True
5,Browser,29.762710,3.031416e-03,12,True
6,Weekend,7.477999,6.245745e-03,1,True
7,Region,6.646898,5.751637e-01,8,False


### Categorical Screening Insights:
* **Statistically Significant ($p < 0.01$):** `Month`, `TrafficType`, `VisitorType_Weekend`, `VisitorType`, `OperatingSystems`, `Browser`, and `Weekend` all show statistically significant associations with customer purchases.
* **Statistically Insignificant ($p = 0.575 > 0.05$):** `Region` fails the significance test. One-hot encoding `Region` creates 9 dummy features that contribute no measurable predictive signal. It is flagged as a candidate for exclusion.

## 4. Cross-Validated Permutation Feature Importance (Numeric Baseline)

Permutation feature importance measures the decrease in validation F1-score when a feature's values are randomly shuffled on held-out validation folds.
* **Model:** `RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)`.
* **Evaluation Protocol:** 5-Fold Stratified K-Fold CV, `scoring='f1'`, `n_repeats=10`.

In [5]:
selection_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

importance = cross_validated_permutation_importance(
    selection_model, X_train_numeric, y_train, scoring='f1', cv=cv, n_repeats=10
)
importance

,feature,importance_mean,importance_std
0,PageValues,0.472295,0.013443
1,ExitRates,0.035904,0.009059
2,BounceRates,0.009004,0.007769
3,SpecialDay,0.005295,0.004552
4,AvgTimePerPage,0.004789,0.002367
5,ProductRelated_Duration,0.004647,0.009732
6,TotalDuration,0.004565,0.007089
7,TotalPages,-0.000654,0.007388
8,ProductRelated,-0.002083,0.005661
9,Informational_Duration,-0.004225,0.003625


### Permutation Importance Findings:
1. **`PageValues` is the dominant feature** (importance: 0.4723), heavily driving the model's purchase predictions.
2. **`ExitRates` is the #2 most important numeric feature** (importance: 0.0359), substantially above `BounceRates` (0.0090).
3. **Negative/Zero Importance:** 6 features (`TotalPages`, `ProductRelated`, `Informational_Duration`, `Administrative`, `Informational`, `Administrative_Duration`) exhibit negative importance. Shuffling them did not degrade validation performance, suggesting redundancy or noise.

## 5. Systematic Empirical Ablation Experiments

We now use `compare_feature_drop` across identical 5-fold splits to rigorously test candidate drop hypotheses.

In [6]:
# Candidate 1: BounceRates and ExitRates redundancy hypothesis
exit_rates_comparison = compare_feature_drop(
    selection_model, X_train_numeric, y_train, 'ExitRates', scoring='f1', cv=cv
)
exit_rates_comparison

,feature_set,mean_score,std_score
0,keep,0.659773,0.011748
1,drop,0.655257,0.012153


**Candidate 1 Outcome:** Dropping `ExitRates` reduces the cross-validated F1 score from **0.6598 to 0.6553** (a loss of ~0.45%). Since `ExitRates` has the second highest permutation importance, **`ExitRates` must be retained**. Correlation alone does not justify discarding valuable predictive variance.

### Candidate 2: Dropping Near-Perfect Collinear Duplicates
`ProductRelated` and `ProductRelated_Duration` are over 98.6% correlated with the newly engineered `TotalPages` and `TotalDuration`. We evaluate dropping the raw redundant pair in favor of the engineered totals.

In [7]:
collinear_cols = ['ProductRelated', 'ProductRelated_Duration']
collinear_comparison = compare_feature_drop(
    selection_model,
    X_train_numeric,
    y_train,
    collinear_cols,
    scoring='f1',
    cv=cv,
    drop_name='drop_collinear_prod',
)
collinear_comparison

,feature_set,mean_score,std_score
0,keep,0.659773,0.011748
1,drop_collinear_prod,0.668188,0.010918


**Candidate 2 Outcome:** Dropping `['ProductRelated', 'ProductRelated_Duration']` **increases the cross-validated F1-score from 0.6598 to 0.6682 (+0.84% improvement)** and reduces cross-fold standard deviation (from 0.0117 to 0.0109). Removing the near-identical raw duplicates eliminates harmful multicollinearity while preserving total browsing metrics (`TotalPages`, `TotalDuration`).

## 6. Sensitivity Analysis: Feature Importance WITHOUT `PageValues`

In EDA (Member 4), `PageValues` was flagged for potential data leakage or unavailability during real-time, early-session inference. We re-run permutation importance without `PageValues` to understand which features drive predictions in early-session / cold-start conditions.

In [8]:
X_no_pv = X_train_numeric.drop(columns=['PageValues'])
imp_no_pv = cross_validated_permutation_importance(
    selection_model, X_no_pv, y_train, scoring='f1', cv=cv, n_repeats=5
)
imp_no_pv

,feature,importance_mean,importance_std
0,TotalDuration,0.155914,0.015519
1,ProductRelated_Duration,0.155657,0.027479
2,ProductRelated,0.126171,0.016922
3,TotalPages,0.107756,0.016583
4,ExitRates,0.104159,0.021215
5,AvgTimePerPage,0.089671,0.027398
6,BounceRates,0.074616,0.017689
7,Administrative,0.042472,0.006873
8,Administrative_Duration,0.033809,0.014121
9,SpecialDay,0.013461,0.004616


### Sensitivity Analysis Insights:
When `PageValues` is absent, **browsing engagement duration (`TotalDuration`), page volume (`TotalPages`), and session progression metrics (`ExitRates`, `AvgTimePerPage`, `BounceRates`)** emerge as the primary predictors. This confirms that behavioral session features carry significant signal when Google Analytics value metrics are not available.

## 7. Final Feature Selection Decision & Pipeline Handoff

Based on our statistical and empirical findings:
1. **Numeric Feature Set:** Drop collinear `ProductRelated` and `ProductRelated_Duration` (gains +0.84% F1). Retain `ExitRates` (dropping hurts F1). Retain all other 11 numeric features (`PageValues`, `BounceRates`, `ExitRates`, `SpecialDay`, `AvgTimePerPage`, `TotalPages`, `TotalDuration`, `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`).
2. **Categorical Feature Set:** Exclude `Region` (Chi-Square $p = 0.575$, statistically insignificant, saving 9 sparse dummy columns). Retain 7 significant categorical features (`Month`, `OperatingSystems`, `Browser`, `TrafficType`, `VisitorType`, `Weekend`, `VisitorType_Weekend`).

In [9]:
# Selected numeric columns (dropping collinear duplicates ProductRelated & ProductRelated_Duration)
selected_numeric_columns = [
    col for col in numeric_columns
    if col not in ['ProductRelated', 'ProductRelated_Duration']
]

# Selected categorical columns (excluding statistically insignificant Region)
selected_categorical_columns = [
    col for col in categorical_columns
    if col != 'Region'
]

print('Selected Numeric Features (11):', selected_numeric_columns)
print('Selected Categorical Features (7):', selected_categorical_columns)
print(f'Total Features Retained: {len(selected_numeric_columns) + len(selected_categorical_columns)} (11 numeric + 7 categorical)')

Selected Numeric Features (11): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgTimePerPage']
Selected Categorical Features (7): ['Month', 'OperatingSystems', 'Browser', 'TrafficType', 'VisitorType', 'Weekend', 'VisitorType_Weekend']
Total Features Retained: 18 (11 numeric + 7 categorical)


**Justification (Member 3):**
* **Statistical Screening:** Chi-Square screening on the training split confirmed `Region` has no significant bivariate relationship with `Revenue` ($p = 0.575 > 0.05$), justifying its removal to eliminate 9 uninformative one-hot dimensions.
* **Empirical Drop Testing:** `ExitRates` was retained because cross-validated ablation proved dropping it degraded F1 from 0.6598 to 0.6553. Conversely, dropping `ProductRelated` and `ProductRelated_Duration` resolved near-perfect collinearity ($r > 0.986$) with `TotalPages` and `TotalDuration`, boosting F1 from 0.6598 to 0.6682.
* **Leakage Prevention:** All selection decisions and model evaluations were executed strictly using 5-Fold Stratified Cross-Validation on training folds. The selected feature lists must now be used by Member 2 to configure the `ColumnTransformer` (fitting transformers only on `X_train`).